# 🔴 Solution: GPT-2 Transformer Block（参考版）

## 📋 题目描述

**难度：** Hard

实现 GPT-2 Transformer 块（nn.Module）。

GPT-2 块使用 pre-norm 架构：在因果自注意力和 MLP 之前进行 LayerNorm，两者都有残差连接。

**签名:** `GPT2Block(d_model, num_heads)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (B, S, d_model)

**返回:** 输出张量 (B, S, d_model)

**约束:**
- Pre-norm：`x = x + attn(ln1(x))`，`x = x + mlp(ln2(x))`
- MLP：Linear(d, 4d) -> GELU -> Linear(4d, d)
- 注意力必须是因果的（未来 token 不能影响过去）

**提示：** Pre-norm 残差：`x = x + attn(ln1(x))`，`x = x + mlp(ln2(x))`。MLP：`Linear(d,4d) → GELU → Linear(4d,d)`。注意力必须是因果的（用 `-inf` 遮蔽未来）。


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def gpt2_block(x: torch.Tensor, ln1_w: torch.Tensor, ln1_b: torch.Tensor, W_q: torch.Tensor, W_k: torch.Tensor, W_v: torch.Tensor, W_o: torch.Tensor, ln2_w: torch.Tensor, ln2_b: torch.Tensor, W_fc1: torch.Tensor, b_fc1: torch.Tensor, W_fc2: torch.Tensor, b_fc2: torch.Tensor, num_heads: int, mask: torch.Tensor = None) -> torch.Tensor:
    # GPT-2 Transformer Block = Self-Attention + FFN + LayerNorm + Residual
    # Pre-Norm 架构：LayerNorm 在注意力/FFN 之前（与原始 Transformer 的 Post-Norm 不同）

    batch_size, seq_len, d_model = x.shape
    d_k = d_model // num_heads

    # ============ 子层 1：Multi-Head Self-Attention ============

    # ---- Step 1.1: Pre-Norm ----
    # LayerNorm 在注意力之前，有助于训练稳定性
    mean1 = x.mean(dim=-1, keepdim=True)
    var1 = ((x - mean1) ** 2).mean(dim=-1, keepdim=True)
    x_norm1 = (x - mean1) / torch.sqrt(var1 + 1e-5)
    x_ln1 = ln1_w * x_norm1 + ln1_b

    # ---- Step 1.2: QKV 投影 ----
    Q = x_ln1 @ W_q
    K = x_ln1 @ W_k
    V = x_ln1 @ W_v

    # ---- Step 1.3: 拆分多头 ----
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)

    # ---- Step 1.4: 注意力计算 ----
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    attn_out = attn @ V

    # ---- Step 1.5: 合并多头 + 输出投影 ----
    attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    attn_out = attn_out @ W_o

    # ---- Step 1.6: 残差连接 ----
    # 残差：让梯度直接流过，缓解深层网络的梯度消失
    x = x + attn_out

    # ============ 子层 2：Feed-Forward Network ============

    # ---- Step 2.1: Pre-Norm ----
    mean2 = x.mean(dim=-1, keepdim=True)
    var2 = ((x - mean2) ** 2).mean(dim=-1, keepdim=True)
    x_norm2 = (x - mean2) / torch.sqrt(var2 + 1e-5)
    x_ln2 = ln2_w * x_norm2 + ln2_b

    # ---- Step 2.2: FFN ----
    # GPT-2 FFN: Linear → GELU → Linear
    # 通常中间维度是 d_model 的 4 倍
    hidden = x_ln2 @ W_fc1 + b_fc1
    hidden = torch.nn.functional.gelu(hidden)
    ffn_out = hidden @ W_fc2 + b_fc2

    # ---- Step 2.3: 残差连接 ----
    x = x + ffn_out

    return x

In [ ]:
block = GPT2Block(64, 4)
print('Output:', block(torch.randn(2, 8, 64)).shape)
print('Params:', sum(p.numel() for p in block.parameters()))

## 📝 核心思路总结

1. **Pre-Norm 架构**：LayerNorm 在子层之前，训练更稳定
2. **残差连接**：x + sublayer(x)，梯度的"高速公路"
3. **两个子层**：Self-Attention（建模关系）+ FFN（存储知识）
4. **GELU 激活**：比 ReLU 更平滑，GPT-2 的标准选择